In [6]:
import pandas as pd


def _load_model_df(model, iterations, reranked, results_dir, naiverag=False, drop_na_sources=True):
    rag_path = f"{results_dir}/{model}/results_{model}_medqa_rag.csv"
    baseline_path = f"{results_dir}/{model}/results_{model}_medqa_base.csv"

    rag_df = pd.read_csv(rag_path)

    if drop_na_sources:
        rag_df = rag_df.dropna(subset=['used_sources'])

    baseline_df = pd.read_csv(baseline_path)

    rag_df = rag_df[rag_df.retrieval_iterations <= iterations]

    baseline_df['llm_response'] = (
        baseline_df['llm_response']
        .str.extract(r'^\s*([A-Za-z])')[0]
        .str.upper()
    )

    rag_df['llm_response'] = (
        rag_df['llm_response']
        .str.extract(r'^\s*([A-Za-z])')[0]
        .str.upper()
    )

    rag_df = rag_df[rag_df['answer_source'] == 'agentic_rag']

    merged = pd.merge(baseline_df, rag_df, on='question_id', suffixes=('_baseline', '_rag'))

    if naiverag:
        naiverag_path = f"{results_dir}/results_{model}_naiverag_reranked.csv"
        naiverag_df = pd.read_csv(naiverag_path)
        naiverag_df['llm_response'] = (
            naiverag_df['llm_response']
            .str.extract(r'^\s*([A-Za-z])')[0]
            .str.upper()
        )
        naiverag_df = naiverag_df.rename(columns={
            'llm_response': 'llm_response_naive',
            'correct_choice': 'correct_choice_naive',
        })
        merged = pd.merge(
            merged,
            naiverag_df[['question_id', 'llm_response_naive', 'correct_choice_naive']],
            on='question_id',
        )

    return merged


def compare_models(
    models,
    iterations=3,
    reranked=True,
    results_dir="../results",
    matched=True,
    naiverag=False,
    drop_na_sources=True,
):
    model_dfs = {
        m: _load_model_df(
            m,
            iterations,
            reranked,
            results_dir,
            naiverag=naiverag,
            drop_na_sources=drop_na_sources,
        )
        for m in models
    }

    if matched:
        common_ids = set.intersection(
            *[set(df['question_id']) for df in model_dfs.values()]
        )
        print(f"Common test cases (matched): {len(common_ids)}")
    else:
        common_ids = None

    rows = []

    for model, df in model_dfs.items():

        if matched:
            df = df[df['question_id'].isin(common_ids)]
            n = len(common_ids)
        else:
            n = len(df)

        baseline_correct = (df['llm_response_baseline'] == df['correct_choice_baseline']).sum()
        rag_correct = (df['llm_response_rag'] == df['correct_choice_rag']).sum()

        baseline_acc = baseline_correct / n
        rag_acc = rag_correct / n

        row = {
            "Model": model,
            "N": n,
            "Baseline Accuracy": baseline_acc,
            "RAG Accuracy": rag_acc,
            "Absolute Gain": rag_acc - baseline_acc,
        }

        if naiverag:
            naive_correct = (df['llm_response_naive'] == df['correct_choice_naive']).sum()
            naive_acc = naive_correct / n
            row["Naive RAG Accuracy"] = naive_acc
            row["Naive RAG Gain"] = naive_acc - baseline_acc

        rows.append(row)

    results_df = pd.DataFrame(rows)

    pct_cols = ["Baseline Accuracy", "RAG Accuracy", "Absolute Gain"]
    if naiverag:
        pct_cols += ["Naive RAG Accuracy", "Naive RAG Gain"]

    for col in pct_cols:
        results_df[col] = (results_df[col] * 100).round(1)

    results_df = results_df.sort_values(by="RAG Accuracy", ascending=False).reset_index(drop=True)

    return results_df


models = ["gpt5", "llama4", "mistrallarge2", "qwen2.5", "gemma3"]
results_table = compare_models(
    models,
    iterations=3,
    reranked=True,
    matched=True,
    naiverag=False,
    drop_na_sources=False,
)

results_table

Common test cases (matched): 2207


,Model,N,Baseline Accuracy,RAG Accuracy,Absolute Gain
0,gpt5,2207,92.7,93.0,0.3
1,llama4,2207,90.4,91.9,1.4
2,qwen2.5,2207,85.3,89.2,3.9
3,mistrallarge2,2207,84.9,86.7,1.8
4,gemma3,2207,76.9,85.5,8.6


In [5]:
import pandas as pd
d = pd.read_csv("../results/results_mistrallarge2_qwenagent_reranked_rag.csv")
d[d.question_id == 8368]

,question_id,question,answer_options,correct_choice,llm_response,llm_response_raw,used_sources,answer_source,conditions,retrieval_iterations,final_doc_count,final_sufficient,final_confidence,retrieval_history,retrieved_documents
1863,8368,Calciphylaxis is seen in CKD patients due to?,"{""A"": ""Warfarin"", ""B"": ""Hypoparathyroidism"", ""...",A,N,Answer: None of the above\nUsed sources: [Doc...,"[Document ID 1], [Document ID 3], [Document ID 8]",agentic_rag,"[""calciphylaxis"", ""chronic kidney disease"", ""C...",3,10,True,0.85,"[{""iteration"": 1, ""query"": [""What is the patho...","[""In chronic renal failure, decreased clearanc..."


In [10]:
import pandas as pd
import json

df = pd.read_csv("../results/faithfulness_llama4.csv")

def citation_precision(doc_ratings_json):
    try:
        ratings = json.loads(doc_ratings_json)
        if not ratings:
            return float("nan")
        return sum(1 for d in ratings if d["rating"] in ("Supports", "Partially supports")) / len(ratings)
    except Exception:
        return float("nan")

df["citation_precision"] = df["doc_ratings"].apply(citation_precision)
df["is_correct"] = df["is_correct"].astype(bool)

# Summary metrics
print(f"N cases:               {len(df)}")
print(f"Mean citation precision: {df['citation_precision'].mean():.3f}")
print(f"\nFaithfulness breakdown:")
print(df["faithfulness"].value_counts())
print(f"\nFaithfulness rate (Yes): {(df['faithfulness'] == 'Yes').mean():.3f}")

# Faithfulness × Correctness cross-tab
display(pd.crosstab(df["faithfulness"], df["is_correct"], 
            colnames=["is_correct"], rownames=["faithfulness"],
            margins=False))

# Citation precision by correctness
df.groupby("is_correct")["citation_precision"].agg(["mean", "count"]).round(3)


N cases:               80
Mean citation precision: 0.706

Faithfulness breakdown:
faithfulness
Yes          49
No           21
Partially    10
Name: count, dtype: int64

Faithfulness rate (Yes): 0.613


is_correct,False,True
faithfulness,,
No,4,17
Partially,1,9
Yes,1,48


,mean,count
is_correct,,
False,0.417,6
True,0.730,74


In [11]:
ct = pd.crosstab(df["faithfulness"], df["is_correct"],
                 colnames=["Correct"], rownames=["Faithfulness"])

ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100  # row percentages

ct_fmt = ct.astype(str) + " (" + ct_pct.round(1).astype(str) + "%)"
ct_fmt


Correct,False,True
Faithfulness,,
No,4 (19.0%),17 (81.0%)
Partially,1 (10.0%),9 (90.0%)
Yes,1 (2.0%),48 (98.0%)


In [66]:
r = pd.read_csv("../results/results_gpt5_qwenagent_reranked_rag.csv")
b = pd.read_csv("../results/results_gpt5.csv")
multiple_answers = r[~r["llm_response"].str.contains(r"[A-Za-z]", na=False)]
multiple_answers

,question_id,question,answer_options,correct_choice,llm_response,llm_response_raw,used_sources,answer_source,conditions,retrieval_iterations,final_doc_count,final_sufficient,final_confidence,retrieval_history,retrieved_documents
221,1486,"When patient is on isotretinoin therapy, monit...","{""A"": ""Liver function test"", ""B"": ""Lipid profi...",B,',"Answer: 'A, B'\nUsed sources: [Document ID 1],...","[Document ID 1], [Document ID 2], [Document ID...",agentic_rag,"[""isotretinoin"", ""monitoring"", ""laboratory tes...",1,10,True,0.95,"[{""iteration"": 1, ""query"": [""What laboratory t...","[""[17]\n\n# Clinical question 10: What are the..."
1047,4930,Which of the following is/are tuberculides:,"{""A"": ""Lichen scrofulosorum"", ""B"": ""Lichen nit...",A,',"Answer: 'A, D'\nUsed sources: [Document ID 1],...","[Document ID 1], [Document ID 8]",agentic_rag,"[""tuberculides"", ""cutaneous tuberculosis"", ""hy...",1,10,True,0.95,"[{""iteration"": 1, ""query"": [""What are tubercul...","[""Fatal disease caused by generalized BCG tube..."
1158,5438,True about malignant melanoma,"{""A"": ""Lymphatic spread"", ""B"": ""Lymph node bio...",A,',"Answer: 'A, C, D'\nUsed sources: [Document ID ...","[Document ID 2], [Document ID 7]",agentic_rag,"[""malignant melanoma"", ""melanoma pathology"", ""...",1,10,True,0.95,"[{""iteration"": 1, ""query"": [""What are the char...","[""# Tumorigenesis and Tumor Progression\n\nFiv..."
1179,5497,Leprosy causes?,"{""A"": ""Rapidly progressive GN"", ""B"": ""Focal gl...",B,',Answer: 'Membranoproliferative GN'\nUsed sourc...,"[Document ID 3], [Document ID 7]",agentic_rag,"[""leprosy"", ""Mycobacterium leprae"", ""infectiou...",1,10,True,0.95,"[{""iteration"": 1, ""query"": ""Leprosy causes?"", ...","[""# POSITION STATEMENT\n\n# Position statement..."
1325,6075,Grisefulvin given for the treatment of fungal ...,"{""A"": ""4 weeks"", ""B"": ""6 weeks"", ""C"": ""2 month...",D,',Answer: '3 months'\nUsed sources: [Document ID...,"[Document ID 5], [Document ID 3], [Document ID 6]",agentic_rag,"[""griseofulvin"", ""fingernail dermatophytosis"",...",1,10,True,0.95,"[{""iteration"": 1, ""query"": [""What is the recom...","[""with a predilection for the ventral nail and..."
1349,6163,PUVA therapy is used in:,"{""A"": ""Psoriasis"", ""B"": ""Lichen planus"", ""C"": ...",A,',"Answer: 'A, B'\nUsed sources: [Document ID 1],...","[Document ID 1], [Document ID 5]",agentic_rag,"[""PUVA therapy"", ""psoriasis"", ""vitiligo"", ""cut...",2,10,True,0.85,"[{""iteration"": 1, ""query"": ""PUVA therapy is us...","[""According to the penetration characteristics..."
1395,6339,Prognosis of malignant melanoma depends upon: ...,"{""A"": ""Grade of tumour"", ""B"": ""Age of the pati...",C,',"Answer: 'B, C, D'\nUsed sources: [Document ID ...","[Document ID 2], [Document ID 4], [Document ID...",agentic_rag,"[""malignant melanoma"", ""prognosis"", ""Breslow t...",1,10,True,0.90,"[{""iteration"": 1, ""query"": [""What factors infl...","[""In clinically node-negative stage I or II pa..."
1396,6347,Cicatricial alopecia is seen in-,"{""A"": ""DLE"", ""B"": ""Psoriasis"", ""C"": ""Alopecia ...",A,',Answer: 'A and D'\nUsed sources: [Document ID ...,"[Document ID 1], [Document ID 2], [Document ID...",agentic_rag,"[""cicatricial alopecia"", ""associated condition...",1,10,True,0.85,"[{""iteration"": 1, ""query"": ""Cicatricial alopec...","[""[263,264] Clinically, there is effacement of..."
1403,6384,Pyoderma gangrenosum is seen in :,"{""A"": ""Crohns disease"", ""B"": ""Diveuculosis"", ""...",C,',"Answer: 'A, C'\nUsed sources: [Document ID 2],...","[Document ID 2], [Document ID 4], [Document ID...",agentic_rag,"[""Pyoderma gangrenosum"", ""associated condition...",2,10,True,0.95,"[{""iteration"": 1, ""query"": ""Pyoderma gangrenos...","[""Localized skin infections, usually staphyloc..."
1429,6506,A person with recurrent oral ulcers with yello...,"{""A"": ""Behcet's syndrome"", ""B"": ""Pemphigus"", ""...",A,',Answer: 'A. Behcet's syndrome'\nUsed 

In [2]:
import numpy as np
from statsmodels.stats.contingency_tables import mcnemar

def calculate_confidence_interval(merged_df, n_bootstrap=1000, ci=95, seed=42):

    rng = np.random.default_rng(seed)
    n = len(merged_df)
    
    baseline_correct = (merged_df.llm_response_gpt == merged_df.correct_choice_gpt).values
    rag_correct = (merged_df.llm_response_rag == merged_df.correct_choice_rag).values
    
    baseline_scores = []
    rag_scores = []
    
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        baseline_scores.append(baseline_correct[idx].mean())
        rag_scores.append(rag_correct[idx].mean())
    
    alpha = (100 - ci) / 2
    baseline_ci = np.percentile(baseline_scores, [alpha, 100 - alpha])
    rag_ci = np.percentile(rag_scores, [alpha, 100 - alpha])
    
    baseline_mean = baseline_correct.mean()
    rag_mean = rag_correct.mean()
    
    return {
        "baseline": {
            "mean": baseline_mean,
            "ci_lower": baseline_ci[0],
            "ci_upper": baseline_ci[1],
            "error": (baseline_ci[1] - baseline_ci[0]) / 2
        },
        "rag": {
            "mean": rag_mean,
            "ci_lower": rag_ci[0],
            "ci_upper": rag_ci[1],
            "error": (rag_ci[1] - rag_ci[0]) / 2
        }
    }


def mcnemar_test_from_df(df, 
                        gpt_col="llm_response_gpt", 
                        rag_col="llm_response_rag",
                        gpt_gt_col="correct_choice_gpt",
                        rag_gt_col="correct_choice_rag",
                        exact=True,
                        verbose=True):

    # correctness vectors
    gpt_correct = df[gpt_col] == df[gpt_gt_col]
    rag_correct = df[rag_col] == df[rag_gt_col]

    # contingency counts
    n01 = np.sum((~gpt_correct) & (rag_correct))  # GPT wrong, RAG correct
    n10 = np.sum((gpt_correct) & (~rag_correct))  # GPT correct, RAG wrong

    table = [[0, n01],
             [n10, 0]]

    # McNemar test
    result = mcnemar(table, exact=exact)

    # accuracies
    gpt_acc = np.mean(gpt_correct)
    rag_acc = np.mean(rag_correct)

    output = {
        "n01 (GPT wrong, RAG correct)": int(n01),
        "n10 (GPT correct, RAG wrong)": int(n10),
        "statistic": float(result.statistic),
        "p_value": float(result.pvalue),
        "gpt_accuracy": float(gpt_acc),
        "rag_accuracy": float(rag_acc),
    }

    if verbose:
        print(f"Baseline accuracy: {gpt_acc:.4f}")
        print(f"RAG accuracy:      {rag_acc:.4f}")
        print(f"n01 (GPT→wrong, RAG→correct): {n01}")
        print(f"n10 (GPT→correct, RAG→wrong): {n10}")
        print(f"p-value: {result.pvalue:.6f}")

        if result.pvalue < 0.05:
            if n01 > n10:
                print("→ RAG is significantly better")
            elif n10 > n01:
                print("→ Baseline is significantly better")
            else:
                print("→ Significant but symmetric (rare case)")
        else:
            print("→ No statistically significant difference")

    return output

In [8]:
import json
INPUT_PATH = "/home/t252a/data/derma/medqa_derma_final.json"
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)
sources = []
for d in data['questions']:
    sources.append(data['questions'][d]['source'])
import pandas as pd
pd.value_counts(sources)

/tmp/ipykernel_1557418/2026629757.py:9: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(sources)
/tmp/ipykernel_1557418/2026629757.py:9: FutureWarning: value_counts with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  pd.value_counts(sources)


MedMCQA                       3412
MedQA                          827
mmlu_medqa                     584
IMPP                            13
mmlu_professional_medicine      11
mmlu_clinical_knowledge          8
Name: count, dtype: int64

In [95]:
import json
import random

INPUT_PATH = "/home/t252a/data/derma/medqa_derma_final.json"
OUTPUT_PATH = "/home/t252a/data/derma/medqa_derma_test.json"
N = 3

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

questions = data.get("questions")
if not isinstance(questions, dict):
    raise ValueError('Expected JSON structure: {"questions": { ... }}')

n = min(N, len(questions))

# sample keys, then rebuild dict
sampled_keys = random.sample(list(questions.keys()), n)
sampled_questions = {k: questions[k] for k in sampled_keys}

out = {"questions": sampled_questions}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2, ensure_ascii=False)

print(f"Saved {n} questions to {OUTPUT_PATH}")

Saved 3 questions to /home/t252a/data/derma/medqa_derma_test.json


In [ ]:
import json
import random

INPUT_PATH = "/home/t252a/data/derma/medqa_derma_final.json"
OUTPUT_PATH = "/home/t252a/data/derma/medqa_derma_test.json"
N = 3

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

questions = data.get("questions")

sampled_questions = {}
for qid, qdata in questions.items():
    if qid in ['1592', '5511', '2808', '3614']:
        sampled_questions[qid] = qdata

out = {"questions": sampled_questions}

#with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
#    json.dump(out, f, indent=2, ensure_ascii=False)

print(f"Saved questions to {OUTPUT_PATH}")

Saved questions to /home/t252a/data/derma/medqa_derma_test.json


In [ ]:
import json
from IPython.display import Markdown, display
with open("../results/retrieved_docs/retrieved_docs_snowflake.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
for d in data:
    if d['question_id'] == '2808':
        instance = d
        break


In [64]:
def call_llm(client, prompt: str) -> str:
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "user", "content": prompt},
        ],
    )
    return response.choices[0].message.content

docs_text = "\n\n".join(
    [f"Doc {i+1}: {doc[:500]}..." for i, doc in enumerate(instance['retrieval_result']['documents'])]
)

prompt = f"""
Evaluate if the retrieved documents contain sufficient information to answer this question.

Return a list of the retrieved documents that are relevant to answering the question, and a brief explanation for why they are relevant. If none of the documents are relevant, return an empty list and an explanation.

QUESTION: {instance['question']}

RETRIEVED DOCUMENTS:
{docs_text}
"""
from together import Together
client = Together(api_key="ed7194518d83152940b72d26d6dcaa01a3a757e97ec466362c6fd7dae896c8b0")
message = call_llm(client, prompt)

In [65]:
Markdown(message)

**Relevant documents**

| Document | Why it is relevant |
|----------|--------------------|
| **Doc 8** | Contains the phrase “Suprabasal acantholytic dermatoses …” – directly discusses the category of skin disorders that produce suprabasal acantholytic blisters and therefore is likely to list the conditions (e.g., pemphigus vulgaris, Hailey‑Hailey disease). |
| **Doc 4** | Section titled “Acantholytic Disorders of the Skin” lists Darier‑White disease, Grover disease, and **Hailey‑Hailey disease** – disorders known to show suprabasal acantholysis, making the document pertinent to the question. |
| **Doc 10** | Provides a detailed overview of “Acantholysis” with entries such as “in pemphigus,” “in Darier‑White disease,” and “in Grover disease,” which are classic examples of suprabasal acantholytic blistering conditions. |

These three documents discuss suprabasal acantholysis and the diseases that exhibit this histologic pattern, and therefore they contain information needed to answer the question.

In [4]:
# GPT-OSS-120B
import pandas as pd
gpt_df = pd.read_csv("../results/results_gptoss120b.csv")
rag_df = pd.read_csv("../results/results_gptoss120b_rag.csv")
rag_df_sufficient = rag_df[rag_df.answer_source == "agentic_rag"]
merged_df_sufficient = pd.merge(gpt_df, rag_df_sufficient, on="question_id", suffixes=("_gpt", "_rag"))
print("Baseline accuracy:", (merged_df_sufficient.llm_response_gpt == merged_df_sufficient.correct_choice_gpt).sum() / len(merged_df_sufficient))
print("RAG accuracy:", (merged_df_sufficient.llm_response_rag == merged_df_sufficient.correct_choice_rag).sum() / len(merged_df_sufficient))

Baseline accuracy: 0.9294590643274854
RAG accuracy: 0.9261695906432749


In [38]:
# GPT-5
import pandas as pd
gpt_df = pd.read_csv("../results/results_gpt5.csv")
gpt_df["llm_response"] = gpt_df["llm_response"].str.replace(r"\..*", "", regex=True)
rag_df = pd.read_csv("../results/results_gpt5_rag.csv")
rag_df["llm_response"] = rag_df["llm_response"].str.replace(r"\..*", "", regex=True)
rag_df_sufficient = rag_df[rag_df.answer_source == "agentic_rag"]
merged_df_sufficient = pd.merge(gpt_df, rag_df_sufficient, on="question_id", suffixes=("_gpt", "_rag"))
merged_df_sufficient = merged_df_sufficient[merged_df_sufficient['llm_response_rag'].str.len() == 1]
print("Baseline accuracy:", (merged_df_sufficient.llm_response_gpt == merged_df_sufficient.correct_choice_gpt).sum() / len(merged_df_sufficient))
print("RAG accuracy:", (merged_df_sufficient.llm_response_rag == merged_df_sufficient.correct_choice_rag).sum() / len(merged_df_sufficient))

Baseline accuracy: 0.9298245614035088
RAG accuracy: 0.9254385964912281


In [39]:
gpt5 = calculate_confidence_interval(merged_df_sufficient)
print(gpt5)

{'baseline': {'mean': 0.9298245614035088, 'ci_lower': 0.9199561403508771, 'ci_upper': 0.9396929824561403, 'error': 0.009868421052631582}, 'rag': {'mean': 0.9254385964912281, 'ci_lower': 0.9148391812865497, 'ci_upper': 0.9349415204678363, 'error': 0.010051169590643283}}


In [3]:
# MiniMax2.7
import pandas as pd
gpt_df = pd.read_csv("../results/results_minimax2.7.csv").dropna(subset=["llm_response"])
rag_df = pd.read_csv("../results/results_minimax2.7_reranked_rag.csv").dropna(subset=["llm_response"])
rag_df_sufficient = rag_df[rag_df.answer_source == "agentic_rag"]
merged_df_sufficient = pd.merge(gpt_df, rag_df_sufficient, on="question_id", suffixes=("_gpt", "_rag"))
print("n:", len(merged_df_sufficient))
print("Baseline accuracy:", (merged_df_sufficient.llm_response_gpt == merged_df_sufficient.correct_choice_gpt).sum() / len(merged_df_sufficient))
print("RAG accuracy:", (merged_df_sufficient.llm_response_rag == merged_df_sufficient.correct_choice_rag).sum() / len(merged_df_sufficient))
print("p-value:", mcnemar_test_from_df(merged_df_sufficient, verbose=False)['p_value'])

n: 2672
Baseline accuracy: 0.9098053892215568
RAG accuracy: 0.9217814371257484
p-value: 0.0227676033538807


In [106]:
merged_df_sufficient[merged_df_sufficient['llm_response_rag'].isna()]

,question_id,question_gpt,answer_options_gpt,correct_choice_gpt,llm_response_gpt,answer_source_gpt,agent_strategy_gpt,retrieval_iterations_gpt,final_doc_count_gpt,retrieval_history_gpt,...,answer_options_rag,correct_choice_rag,llm_response_rag,answer_source_rag,agent_strategy_rag,retrieval_iterations_rag,final_doc_count_rag,retrieval_history_rag,retrieved_documents_rag,llm_response_raw_rag
50,75,A 4-year-old girl is brought to the physician ...,"{""A"": ""Potassium hydroxide preparation"", ""B"": ...",D,D,non_rag,NaN,0,0,NaN,...,"{""A"": ""Potassium hydroxide preparation"", ""B"": ...",D,NaN,agentic_rag,"{""collections_to_search"": [""primary"", ""books""]...",1,10,"[{""iteration"": 1, ""docs_retrieved"": 54, ""total...","[""# Acute generalized exanthematous pustulosis...",NaN
151,221,A 57-year-old woman comes to the physician bec...,"{""A"": ""Decrease in elastin fiber assembly"", ""B...",A,E,non_rag,NaN,0,0,NaN,...,"{""A"": ""Decrease in elastin fiber assembly"", ""B...",A,NaN,agentic_rag,"{""collections_to_search"": [""primary"", ""books""]...",1,10,"[{""iteration"": 1, ""docs_retrieved"": 44, ""total...","[""Several of the mechanisms known to be involv...",NaN
198,301,A 47-year-old woman comes to her primary care ...,"{""A"": ""Further questioning"", ""B"": ""Reassurance...",A,D,non_rag,NaN,0,0,NaN,...,"{""A"": ""Further questioning"", ""B"": ""Reassurance...",A,NaN,agentic_rag,"{""collections_to_search"": [""primary"", ""books""]...",1,10,"[{""iteration"": 1, ""docs_retrieved"": 41, ""total...","[""- Medication history: allergy; all prescript...",NaN
223,333,A 4-year-old girl is brought to the emergency ...,"{""A"": ""Notify Child Protective Services"", ""B"":...",A,E,non_rag,NaN,0,0,NaN,...,"{""A"": ""Notify Child Protective Services"", ""B"":...",A,NaN,agentic_rag,"{""collections_to_search"": [""primary"", ""books""]...",1,10,"[{""iteration"": 1, ""docs_retrieved"": 49, ""total...","[""They should provide support for rapid decisi...",NaN
267,408,A 35-year-old female comes to the physician be...,"{""A"": ""Dental caries"", ""B"": ""Antiphospholipid ...",C,A,non_rag,NaN,0,0,NaN,...,"{""A"": ""Dental caries"", ""B"": ""Antiphospholipid ...",C,NaN,agentic_rag,"{""collections_to_search"": [""primary"", ""books""]...",3,10,"[{""iteration"": 1, ""docs_retrieved"": 41, ""total...","[""# Consensus statement on the diagnosis and t...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2448,8524,A 10 year old child experiences recurrent skin...,"{""A"": ""Topoisomerase"", ""B"": ""3' \u2192 5' Exon...",C,C,non_rag,NaN,0,0,NaN,...,"{""A"": ""Topoisomerase"", ""B"": ""3' \u2192 5' Exon...",C,NaN,agentic_rag,"{""collections_to_search"": [""primary"", ""books""]...",1,10,"[{""iteration"": 1, ""docs_retrieved"": 51, ""total...","[""In particular, the importance of DNA as a ta...",NaN
2450,8533,60 year old man presents to his physician afte...,"{""A"": ""Benign nevus cells"", ""B"": ""Malignant ne...",D,D,non_rag,NaN,0,0,NaN,...,"{""A"": ""Benign nevus cells"", ""B"": ""Malignant ne...",D,NaN,agentic_rag,"{""collections_to_search"": [""primary"", ""books""]...",1,10,"[{""iteration"": 1, ""docs_retrieved"": 53, ""total...","[""Xanthomas are rarely seen. This condition is...",NaN
2508,8794,what is true about keloids,"{""A"": ""it appears immediately after surgery"", ...",B,E,non_rag,NaN,0,0,NaN,...,"{""A"": ""it appears immediately after surgery"", ...",B,NaN,agentic_rag,"{""collections_to_search"": [""primary"", ""books""]...",1,10,"[{""iteration"": 1, ""docs_retrieved"": 47, ""total...","[""This is known as a hypertrophic scar in cont...",NaN
2558,9028,"A patient presented with scarring alopecia, th...","{""A"": ""Psoriasis"", ""B"": ""Leprosy"", ""C"": ""Liche...",D,C,non_rag,NaN,0,0,NaN,...,"{""A"": ""Psoriasis"", ""B"": ""Leprosy"", ""C"": ""Liche...",D,NaN,agentic_rag,"{""collections_to_search"": [""primary"", ""books""]...",1,10,"[{""iteration"": 1, ""docs_retrieved"": 48, ""total...","[""# 

In [58]:
# Qwen3-235B
import pandas as pd
gpt_df = pd.read_csv("../results/results_qwen3.csv").dropna(subset=["llm_response"])
gpt_df["llm_response"] = gpt_df["llm_response"].str.replace(r"\..*", "", regex=True)
rag_df = pd.read_csv("../results/results_qwen3_reranked_rag.csv").dropna(subset=["llm_response"])
rag_df["llm_response"] = rag_df["llm_response"].str.replace(r"\..*", "", regex=True)
rag_df_sufficient = rag_df[rag_df.answer_source == "agentic_rag"]
merged_df_sufficient = pd.merge(gpt_df, rag_df_sufficient, on="question_id", suffixes=("_gpt", "_rag"))
print("Baseline accuracy:", ((merged_df_sufficient.llm_response_gpt == merged_df_sufficient.correct_choice_gpt).sum() / len(merged_df_sufficient)).round(4))
print("RAG accuracy:", ((merged_df_sufficient.llm_response_rag == merged_df_sufficient.correct_choice_rag).sum() / len(merged_df_sufficient)).round(4))
print("p-value:", mcnemar_test_from_df(merged_df_sufficient, verbose=False)['p_value'])

Baseline accuracy: 0.9076
RAG accuracy: 0.908
p-value: 0.9999999999999998


In [42]:
# Kimi
import pandas as pd
gpt_df = pd.read_csv("../results/results_kimik2.5.csv").dropna(subset=["llm_response"])
gpt_df["llm_response"] = gpt_df["llm_response"].str.replace(r"\..*", "", regex=True)
rag_df = pd.read_csv("../results/results_kimik2.5_rag.csv").dropna(subset=["llm_response"])
rag_df["llm_response"] = rag_df["llm_response"].str.replace(r"\..*", "", regex=True)
rag_df_sufficient = rag_df[rag_df.answer_source == "agentic_rag"]
merged_df_sufficient = pd.merge(gpt_df, rag_df_sufficient, on="question_id", suffixes=("_gpt", "_rag"))
print("Baseline accuracy:", (merged_df_sufficient.llm_response_gpt == merged_df_sufficient.correct_choice_gpt).sum() / len(merged_df_sufficient))
print("RAG accuracy:", (merged_df_sufficient.llm_response_rag == merged_df_sufficient.correct_choice_rag).sum() / len(merged_df_sufficient))

Baseline accuracy: 0.90625
RAG accuracy: 1.0


In [49]:
# GPT-5-mini
import pandas as pd
gpt_df = pd.read_csv("../results/results_gpt5mini.csv").dropna(subset=["llm_response"])
rag_df = pd.read_csv("../results/results_gpt5mini_rag.csv").dropna(subset=["llm_response"])
rag_df_sufficient = rag_df[rag_df.answer_source == "agentic_rag"]
merged_df_sufficient = pd.merge(gpt_df, rag_df_sufficient, on="question_id", suffixes=("_gpt", "_rag"))
print("Baseline accuracy:", (merged_df_sufficient.llm_response_gpt == merged_df_sufficient.correct_choice_gpt).sum() / len(merged_df_sufficient))
print("RAG accuracy:", (merged_df_sufficient.llm_response_rag == merged_df_sufficient.correct_choice_rag).sum() / len(merged_df_sufficient))
print("p-value:", mcnemar_test_from_df(merged_df_sufficient, verbose=False)['p_value'])

Baseline accuracy: 0.8863304093567251
RAG accuracy: 0.8914473684210527
p-value: 0.37644713102655936


In [31]:
gpt5mini = calculate_confidence_interval(merged_df_sufficient)
print(gpt5mini)

{'baseline': {'mean': 0.8863304093567251, 'ci_lower': 0.874625365497076, 'ci_upper': 0.8972953216374269, 'error': 0.011334978070175417}, 'rag': {'mean': 0.8907163742690059, 'ci_lower': 0.8801169590643275, 'ci_upper': 0.9024122807017544, 'error': 0.011147660818713434}}


In [10]:
# DeepSeek-V3.1
import pandas as pd
gpt_df = pd.read_csv("../results/results_deepseekv3.1.csv").dropna(subset=["llm_response"])
gpt_df["llm_response"] = gpt_df["llm_response"].str.replace(r"\..*", "", regex=True)

rag_df = pd.read_csv("../results/results_deepseekv3.1_reranked_rag.csv").dropna(subset=["llm_response"]) 
rag_df["llm_response"] = rag_df["llm_response"].str.replace(r"\..*", "", regex=True)

rag_df_sufficient = rag_df[rag_df.answer_source == "agentic_rag"]
merged_df_sufficient = pd.merge(gpt_df, rag_df_sufficient, on="question_id", suffixes=("_gpt", "_rag"))

print("Baseline accuracy:", (merged_df_sufficient.llm_response_gpt == merged_df_sufficient.correct_choice_gpt).sum() / len(merged_df_sufficient))

print("RAG accuracy:", (merged_df_sufficient.llm_response_rag == merged_df_sufficient.correct_choice_rag).sum() / len(merged_df_sufficient))


Baseline accuracy: 0.853125
RAG accuracy: 0.8796875


In [53]:
# LLama3.3
import pandas as pd
gpt_df = pd.read_csv("../results/results_llama3.3.csv").dropna(subset=["llm_response"])
rag_df = pd.read_csv("../results/results_llama3.3_rag.csv").dropna(subset=["llm_response"])
rag_df_sufficient = rag_df[rag_df.answer_source == "agentic_rag"]
merged_df_sufficient = pd.merge(gpt_df, rag_df_sufficient, on="question_id", suffixes=("_gpt", "_rag"))
print("Baseline accuracy:", (merged_df_sufficient.llm_response_gpt == merged_df_sufficient.correct_choice_gpt).sum() / len(merged_df_sufficient))
print("RAG accuracy:", (merged_df_sufficient.llm_response_rag == merged_df_sufficient.correct_choice_rag).sum() / len(merged_df_sufficient))

Baseline accuracy: 0.9261425959780621
RAG accuracy: 0.8764168190127971


In [12]:
rag_df_insufficient = rag_df[rag_df.answer_source == "agentic_rag_insufficient"]
rag_df_sufficient = rag_df[rag_df.answer_source == "agentic_rag"]
merged_df_sufficient = pd.merge(gpt_df, rag_df_sufficient, on="question_id", suffixes=("_gpt", "_rag"))
merged_df_insufficient = pd.merge(gpt_df, rag_df_insufficient, on="question_id", suffixes=("_gpt", "_rag"))
merged_df_sufficient[merged_df_sufficient.llm_response_gpt != merged_df_sufficient.llm_response_rag][['question_id', 'correct_choice_rag', 'llm_response_gpt', 'llm_response_rag']]

,question_id,correct_choice_rag,llm_response_gpt,llm_response_rag
31,103,A,B,A
39,119,A,A,D
101,289,A,A,D
102,292,D,D,C
107,304,A,A,D
...,...,...,...,...
1421,9028,D,B,C
1426,9081,C,D,C
1451,9332,A,A,B
1508,9908,A,A,D


In [6]:
(merged_df_sufficient.llm_response_gpt == merged_df_sufficient.correct_choice_rag).sum()

2425

In [34]:
import os
import json
import pandas as pd
from pathlib import Path

def extract_answer_source(obj, filename, rows):
    """
    Recursively search for 'answer_source' keys in a JSON object
    and append results to rows list.
    """
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key == "answer_source":
                rows.append({
                    "file": filename,
                    "answer_source": value
                })
            else:
                extract_answer_source(value, filename, rows)

    elif isinstance(obj, list):
        for item in obj:
            extract_answer_source(item, filename, rows)

def jsons_to_dataframe(directory_path):
    rows = []
    directory = Path(directory_path)

    for file_path in directory.glob("*.json"):
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            extract_answer_source(data, file_path.name, rows)

        except Exception as e:
            print(f"Error reading {file_path.name}: {e}")

    return pd.DataFrame(rows)

directory = "../results/retrieved_docs_medqa_snowflakel/"
df = jsons_to_dataframe(directory)
df.answer_source.value_counts()

answer_source
agentic_rag_insufficient    3341
agentic_rag                 1513
no_documents                   1
Name: count, dtype: int64

In [35]:
import chromadb

CHROMA_DB_PATH = "../chromadb_snowflakev2"

client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

collections = client.list_collections()

if not collections:
    print("No collections found.")
else:
    for collection in collections:
        col = client.get_collection(collection.name)
        print(f"{collection.name}: {col.count()} docs")

eadv_guidelines: 2881 docs
books: 4534 docs


In [6]:
import pandas as pd
df = pd.read_csv("../results/question_categories.csv")
df[df.question_type == 'mechanism'].head(5)

,question_id,disease_category,disease_prevalence,question_type,requires_visual_reasoning,raw_llm_response
6,6,staphylococcal scalded skin syndrome,moderate,mechanism,False,"{\n ""disease_category"": ""staphylococcal scald..."
8,8,herpes zoster,common,mechanism,True,"{\n ""disease_category"": ""herpes zoster"",\n ""..."
11,11,urticaria,common,mechanism,False,"{\n ""disease_category"": ""urticaria"",\n ""dise..."
14,14,xeroderma pigmentosum,rare,mechanism,False,"{\n ""disease_category"": ""xeroderma pigmentosu..."
18,18,bullous pemphigoid,moderate,mechanism,False,"{\n ""disease_category"": ""bullous pemphigoid"",..."


In [16]:
import pandas as pd
df = pd.read_csv("../results/results_gpt5mini_rag.csv")
df[df.question_id == 14].answer_options.iloc[0]

'{"A": "Nucleotide excision repair", "B": "Non-homologous end joining", "C": "Homologous recombination", "D": "Mismatch repair"}'

In [33]:
from pathlib import Path
from datetime import datetime

directory = Path("../results/retrieved_docs_medqa_qwenagent/")
cutoff = datetime(2026, 5, 11, 23, 40, 0)
files = []
for path in directory.rglob("*"):
    if path.is_file():
        mtime = datetime.fromtimestamp(path.stat().st_mtime)
        if mtime > cutoff:
            files.append(path)

In [34]:
len(files)

1154